# Option B — Hybrid Pipeline: GEE (1980–2020) + Local Rasters (2021–2025)

Same structure as your original workflow: historical years from Earth Engine's
PRISM monthly collection, recent years computed locally from your downloaded
`prism_tmean_us_25m_YYYYMM.tif` files, stitched into one 1980–2025 series.

Watersheds: **Navajo, Blue Mesa, Pueblo** (`co_top3_watersheds_combined.shp`).

Requirements in Drive: the 5 shapefile components, and the 15 monthly tifs
(Apr/May/Jun 2021–2025), any folder — everything is found recursively.


In [ ]:
# --- 1. Setup: install + imports + mount Drive ---
!pip -q install geopandas rasterio earthengine-api

import ee
import os, re, glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# --- 2. Authenticate Earth Engine ---
ee.Authenticate()
ee.Initialize(project='esiil-2026-500417')
print('Earth Engine initialized with project esiil-2026-500417')

In [ ]:
# --- 3. Load the top-3 watershed shapefile ---
hits = (glob.glob('/content/drive/MyDrive/**/co_top3_watersheds_combined.shp', recursive=True)
        + glob.glob('/content/co_top3_watersheds_combined.shp'))
assert hits, 'co_top3_watersheds_combined.shp not found — upload all 5 files (.shp .shx .dbf .prj .cpg) to Drive'
SHP = hits[0]
print('Using:', SHP)

watersheds = gpd.read_file(SHP).to_crs(4326)
print(f'{len(watersheds)} watershed(s):', sorted(watersheds['name'].tolist()))

features = [ee.Feature(ee.Geometry(row.geometry.__geo_interface__), {'name': row['name']})
            for _, row in watersheds.iterrows()]
watersheds_fc = ee.FeatureCollection(features)

In [ ]:
# --- 4. GEE: basin-mean tmean, 1980-2020, PRISM monthly ---
# Uses AN81m for these years. Note: AN81m is deprecated and frozen at 2020-12,
# but 1980-2020 is exactly the span it covers, so it works fine here. Swap
# 'OREGONSTATE/PRISM/ANm' in if you prefer the current asset.
COLLECTION = 'OREGONSTATE/PRISM/AN81m'
YEARS_GEE = range(1980, 2021)
MONTHS = [4, 5, 6]
col = ee.ImageCollection(COLLECTION)

gee_rows = []
for y in YEARS_GEE:
    for m in MONTHS:
        start = ee.Date.fromYMD(y, m, 1)
        month_col = col.filterDate(start, start.advance(1, 'month')).select('tmean')
        if month_col.size().getInfo() == 0:
            print(f'{y}-{m:02d}: MISSING in GEE, skipping')
            for name in watersheds['name']:
                gee_rows.append({'watershed': name, 'year': y, 'month': m, 'tmean_c': None})
            continue
        stats = month_col.first().reduceRegions(collection=watersheds_fc,
                                                reducer=ee.Reducer.mean(),
                                                scale=4000).getInfo()
        for feat in stats['features']:
            p = feat['properties']
            gee_rows.append({'watershed': p['name'], 'year': y, 'month': m,
                             'tmean_c': p.get('mean')})
    print(f'{y}: done')

gee_df = pd.DataFrame(gee_rows)
gee_df.to_csv('/content/drive/MyDrive/prism_gee_monthly_1980_2020_TOP3.csv', index=False)
print('\nSaved prism_gee_monthly_1980_2020_TOP3.csv —', len(gee_df), 'rows')

In [ ]:
# --- 5. LOCAL RASTERS: basin-mean tmean, 2021-2025 ---
# Recursively finds your tifs anywhere in Drive, matches by the YYYYMM code
# in the filename, masks each raster to each watershed polygon.
candidates = glob.glob('/content/drive/MyDrive/**/*.tif', recursive=True)
candidates = [p for p in candidates if 'tmean' in os.path.basename(p).lower()]

file_index = {}
for p in candidates:
    mm = re.search(r'(20\d{2})(0[1-9]|1[0-2])', os.path.basename(p))
    if mm:
        file_index[mm.group(1) + mm.group(2)] = p
print(f'Found {len(file_index)} usable monthly raster(s)')

local_rows = []
for y in range(2021, 2026):
    for mth in ('04', '05', '06'):
        ym = f'{y}{mth}'
        path = file_index.get(ym)
        if path is None:
            print(f'{ym}:  MISSING (no matching .tif found in Drive)')
            for name in watersheds['name']:
                local_rows.append({'watershed': name, 'year': y,
                                    'month': int(mth), 'tmean_c': None})
            continue
        with rasterio.open(path) as src:
            nd = src.nodata if src.nodata is not None else -9999
            for _, row in watersheds.iterrows():
                bproj = gpd.GeoSeries([row.geometry], crs=watersheds.crs).to_crs(src.crs)
                geom  = [mapping(bproj.iloc[0])]
                out, _ = mask(src, geom, crop=True, filled=True)
                arr = out[0].astype('float64')
                valid = arr[(arr != nd) & np.isfinite(arr) & (arr > -9990)]
                val = float(valid.mean()) if valid.size else None
                local_rows.append({'watershed': row['name'], 'year': y,
                                    'month': int(mth), 'tmean_c': val})
        print(f'{ym}:  done  <- {path}')

local_df = pd.DataFrame(local_rows)
local_df.to_csv('/content/drive/MyDrive/prism_local_monthly_2021_2025_TOP3.csv', index=False)
print('\nSaved prism_local_monthly_2021_2025_TOP3.csv —', len(local_df), 'rows')

In [ ]:
# --- 6. STITCH: combine GEE (1980-2020) + local (2021-2025) ---
monthly = (pd.concat([gee_df, local_df], ignore_index=True)
             .drop_duplicates(['watershed', 'year', 'month'], keep='last')  # local wins on overlap
             .sort_values(['watershed', 'year', 'month'])
             .reset_index(drop=True))

wide = (monthly.pivot(index=['watershed', 'year'], columns='month', values='tmean_c')
               .rename(columns={4: 'april_c', 5: 'may_c', 6: 'june_c'})
               .reset_index())

mcols = ['april_c', 'may_c', 'june_c']
complete = wide[mcols].notna().sum(axis=1)
wide['temp_aprjun_c'] = wide[mcols].mean(axis=1).where(complete == 3)

wide.to_csv('/content/drive/MyDrive/temp_monthly_1980_2025_TOP3_WATERSHEDS_hybrid.csv', index=False)
print(f'rows: {len(wide)} | watersheds: {wide.watershed.nunique()} | '
      f'years: {wide.year.min()}–{wide.year.max()}')
print(f'missing month-cells: {int(wide[mcols].isna().sum().sum())}')

In [ ]:
# --- 7. Cross-watershed average: one row per year ---
avg = (wide.groupby('year')[mcols].mean()
            .round(4)
            .rename(columns={'april_c': 'April', 'may_c': 'May', 'june_c': 'June'})
            .reset_index())

avg.to_csv('/content/drive/MyDrive/temp_3watersheds_mean_by_year_1980_2025_hybrid.csv', index=False)
print(avg.to_string(index=False))